# End-to-End Pipeline Testing — Code Generation v2.4

Full pipeline: Steps 1–5 (preprocessing) → Step 6 (code generation) → execution

**Required uploads:**
- `sf_film_May7_2025_data.gpkg` — the film locations dataset
- `code_generation_v2.md` — the code generation prompt

**Rate limiting:** Gemini free tier = 10 RPM. LLM calls use an 8s delay.


In [ ]:
!pip install geopandas google-genai shapely


In [ ]:
import json
import re
import time
import copy
import unicodedata
import difflib
import textwrap
from typing import Dict, Any, List, Optional, Tuple
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd

from google import genai
from google.genai import types


In [ ]:
# === CONFIGURATION ===

# Model: Gemini 2.5 Flash      | RPM:5   TPM: 250K  RPD: 20  |  gemini-2.5-flash
# Model: Gemini 2.5 Flash Lite | RPM: 10 TPM: 250K  RPD: 20  |  gemini-2.5-flash-lite
# Model: Gemini 3 Flash        RPM: 5      TPM: 250K     RPD: 20  | gemini-3-flash-preview
# Model: Gemini 3.1 Flash Lite RPM: 0 / 15   TPM: 250K  RPD: 500 |  gemini-3.1-flash-lite-preview


from google.colab import userdata
MODEL_NAME = "gemini-2.5-flash"
MODEL_NAME = "gemini-3-flash-preview"
MODEL_NAME = "gemini-2.5-flash-lite"
MODEL_NAME = "gemini-3.1-flash-lite-preview"
LLM_DELAY = 8  # seconds between LLM calls (rate limit)

# Load API key from Colab Secrets
# To set this up: click the 🔑 icon in the left sidebar → add a secret named GEMINI_API_KEY

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in Colab Secrets. Add it via the 🔑 sidebar."
print("API key loaded successfully.")


client = genai.Client(api_key=GEMINI_API_KEY)


API key loaded successfully.


In [ ]:
# === LOAD DATA ===
from google.colab import drive
drive.mount('/content/drive')

#=== changing the DB_PATH to the latest version after conversion of April-2026 =======================

# DB_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/database/SQLite-stuff/sf_film_May7_2025_data.gpkg'
DB_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/data-reconcillation-2026/sf_film_2026_04_24_data.gpkg'
gdf = gpd.read_file(DB_PATH)

# cast float64 to Int64
gdf['Year'] = gdf['Year'].astype('Int64')
gdf['Supervisor_District'] = gdf['Supervisor_District'].astype('Int64')

assert (gdf['Year'].dropna() % 1 == 0).all()
assert (gdf['Supervisor_District'].dropna() % 1 == 0).all()

print(f"Loaded {len(gdf)} rows, {len(gdf.columns)} columns")
print(f"Columns: {list(gdf.columns)}")
print(f"Unique films: {gdf.drop_duplicates(subset=['Title','Year']).shape[0]}")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 2208 rows, 14 columns
Columns: ['Title', 'Year', 'Locations', 'Fun_Facts', 'Production_Company', 'Distributor', 'Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3', 'Neighborhood', 'Supervisor_District', 'geometry']
Unique films: 352


> ... a post-load sanity cell, right after gpd.read_file(...). Cheap, fast, and if SFgov ever ships a future CSV that the conversion script doesn't catch — or if someone accidentally loads the old gpkg — you'll know on the next run instead of debugging a mysterious "Market Street" query that returns nothing because the query normalized to "Market St" but the data didn't.



In [ ]:
# redundant work on already-normalized data since the conversion of 2026_dataset to gpkg dataset.
# must run to make this happen: Street/Place/Avenue/Boulevard/etc normalizer
# gdf['Locations'] = gdf['Locations'].apply(normalize_street_suffixes)
assert not gdf['Locations'].str.contains(r'\bStreet\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bAvenue\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bBoulevard\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bPlaces?\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bStreets?\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bAvenues?\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bBoulevards?\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bLanes?\b', case=False, na=False).any()
assert not gdf['Locations'].str.contains(r'\bRoads?\b', case=False, na=False).any()
# gdf[gdf['Locations'].str.contains(r'\bPl\.?(?!\w)', case=False, na=False)]['Locations'].head(20)

## Street/Place/Avenue/Boulevard/etc normalizer

In [ ]:
"""
Street suffix normalizer for SF film locations.

One-way transformation from canonical / full forms to short, period-less
abbreviations. Apply this to BOTH sides for matching to work cleanly:

    - the user's query        (call from Step 1 normalizer)
    - the Locations column    (call once at data load time)

Rules covered (6):
    Street / Streets / St. / St      ->  St
    Avenue / Avenues / Ave. / Ave    ->  Ave
    Boulevard / Boulevards / Blvd.   ->  Blvd
    Place / Places / Pl. / Pl        ->  Pl
    Lane  / Lanes   / Ln.  / Ln      ->  Ln
    Road  / Roads   / Rd.  / Rd      ->  Rd

Saint guard:
    'St' and 'St.' are ambiguous — they can mean Street (trailing) or
    Saint (leading). To avoid mangling "St. Germain" into "St Germain"-
    as-a-Street, the abbreviated Street pattern requires a letter
    immediately before the 'S', separated by a single space. This
    correctly rejects:
        "100 St. Germain"    (preceded by digit)
        "St. Francis"        (preceded by start-of-string)
    and correctly accepts:
        "Market St."         (preceded by 't' from Market)
        "1 Front St"         (preceded by 't' from Front)
        "Vallejo st"         (preceded by 'o' from Vallejo)

Non-goals:
    - Drive is NOT normalized. The only 'Dr.' in the data is 'Dr. Carlton
      B. Goodlett Pl.' (Doctor, not Drive). All actual Drive occurrences
      in the dataset are spelled out, so no rule is needed.
"""

import re


# Order matters. Full forms run first (they're unambiguous); the
# abbreviated Street form with its Saint guard runs after. For the
# other suffixes, order is cosmetic but kept consistent.
#
# Each rule is (compiled_pattern, replacement_string).

NORMALIZATION_RULES = [
    # --- Street ---
    # Full forms (Street, Streets) — no ambiguity with Saint.
    (re.compile(r'\bStreets?\b', re.IGNORECASE), 'St'),
    # Abbreviated forms (St, St.) — Saint guard via lookbehind requiring
    # [letter][space] immediately before. (?!\w) ensures we don't match
    # into a longer word like "Stockton".
    (re.compile(r'(?<=[a-zA-Z]\s)St\.?(?!\w)', re.IGNORECASE), 'St'),

    # --- Avenue ---
    (re.compile(r'\bAvenues?\b', re.IGNORECASE), 'Ave'),
    (re.compile(r'\bAve\.?(?!\w)', re.IGNORECASE), 'Ave'),

    # --- Boulevard ---
    (re.compile(r'\bBoulevards?\b', re.IGNORECASE), 'Blvd'),
    (re.compile(r'\bBlvd\.?(?!\w)', re.IGNORECASE), 'Blvd'),

    # --- Place ---
    (re.compile(r'\bPlaces?\b', re.IGNORECASE), 'Pl'),
    (re.compile(r'\bPl\.?(?!\w)', re.IGNORECASE), 'Pl'),

    # --- Lane ---
    (re.compile(r'\bLanes?\b', re.IGNORECASE), 'Ln'),
    (re.compile(r'\bLn\.?(?!\w)', re.IGNORECASE), 'Ln'),

    # --- Road ---
    (re.compile(r'\bRoads?\b', re.IGNORECASE), 'Rd'),
    (re.compile(r'\bRd\.?(?!\w)', re.IGNORECASE), 'Rd'),
]


def normalize_street_suffixes(text):
    """
    Apply all suffix normalization rules to a single string.

    Returns the input with recognized street suffixes converted to
    their short, period-less abbreviations. Leaves 'St.' untouched
    when it is almost certainly 'Saint' (preceded by a digit or
    start-of-string rather than a street name).

    Safe on None or empty input (returns input unchanged).
    """
    if not text:
        return text
    for pattern, replacement in NORMALIZATION_RULES:
        text = pattern.sub(replacement, text)
    return text




## manual testing

In [ ]:
# gdf[gdf['Director'] == 'Nicholas Meyer']

# gdf[gdf['Locations'].str.contains('golden gate bridge', case=False, na=False)]['Title'].nunique()
# print(gdf.columns)
print(gdf.dtypes)

Title                    object
Year                      Int64
Locations                object
Fun_Facts                object
Production_Company       object
Distributor              object
Director                 object
Writer                   object
Actor_1                  object
Actor_2                  object
Actor_3                  object
Neighborhood             object
Supervisor_District       Int64
geometry               geometry
dtype: object


## known values mechanics

In [ ]:
# === KNOWN VALUES (for fuzzy matching) ===
def extract_known_values(gdf):
    known = {}
    for col in ['Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3']:
        vals = gdf[col].dropna().astype(str).str.strip()
        vals = vals[(vals != '') & (~vals.str.lower().isin(['none','nan','null']))]
        known[col] = sorted(vals.unique().tolist())
    known['Actor'] = sorted(set(known.pop('Actor_1') + known.pop('Actor_2') + known.pop('Actor_3')))
    for col in ['Title', 'Locations']:
        vals = gdf[col].dropna().astype(str).str.strip()
        vals = vals[(vals != '') & (~vals.str.lower().isin(['none','nan','null']))]
        known[col] = sorted(vals.unique().tolist())
    return known

known_values = extract_known_values(gdf)
print(f"Known values: { {k: len(v) for k, v in known_values.items()} }")


Known values: {'Director': 290, 'Writer': 307, 'Actor': 661, 'Title': 350, 'Locations': 1749}


In [ ]:
# === LLM HELPER ===
import os
import time

from google.protobuf.json_format import MessageToDict

RAW_RESPONSE_DIR = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/raw_responses'
os.makedirs(RAW_RESPONSE_DIR, exist_ok=True)

def call_gemini(system_prompt, user_input, temperature=0, label=None):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=user_input,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            temperature=temperature,
        ),
    )

    # Capture raw response for post-mortem debugging
    _save_raw_response(response, label)

    return response.text




def _save_raw_response(response, label):
    """Save the raw Gemini response. If dict conversion fails, saves as string."""
    try:
        os.makedirs(RAW_RESPONSE_DIR, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S')
        safe_label = (label or 'unknown').replace('/', '_').replace(' ', '_')
        path = os.path.join(RAW_RESPONSE_DIR, f'{ts}_{safe_label}.json')

        response_data = None

        # Attempt 1: The most common internal dict conversion
        try:
            if hasattr(response, 'to_dict'):
                response_data = response.to_dict()
            elif hasattr(response, '_result'):
                response_data = type(response._result).to_dict(response._result)
        except:
            pass

        # Attempt 2: If it's a proto-message but sneaky
        if not response_data:
            try:
                from google.protobuf.json_format import MessageToDict
                response_data = MessageToDict(response)
            except:
                pass

        with open(path, 'w', encoding='utf-8') as f:
            if response_data:
                # If we successfully got a dict, save as pretty JSON
                json.dump(response_data, f, indent=4)
            else:
                # THE FALLBACK: Save the string representation
                # This ensures you NEVER lose the data even if the object is weird
                f.write(str(response))

        print(f"Successfully saved to {path}")

    except Exception as e:
        print(f'   [Total failure: {repr(e)}]')



## LLM-resiliency stuff

In [ ]:
# =============================================================================
# CELL: LLM resilience layer — paste this into your notebook AFTER the existing
#       call_gemini definition and BEFORE the step functions (extract_filters,
#       decompose_query, generate_code).
#
# This cell does NOT modify call_gemini. It defines:
#   - gemini_provider_call(...)     : adapter that matches the wrapper's signature
#   - call_gemini_safely(...)       : the public entry point step functions will use
#
# After you run this cell, the next step (in a follow-up edit) is to replace
# `call_gemini(...)` calls inside extract_filters / decompose_query /
# generate_code with `call_gemini_safely(...)` and switch them to return the
# {ok, result, error} envelope.
# =============================================================================

# Make sure llm_resilience.py is importable. In Colab, drop the file into
# /content (or your Drive folder) and add it to sys.path:
#
#   import sys
#   sys.path.insert(0, '/content')
#
# Then:
from llm_resilience import (
    GeminiError,
    RetryPolicy,
    call_gemini_with_retry,
)


# -----------------------------------------------------------------------------
# Adapter: bridge your existing call_gemini to the wrapper's expected signature.
# The wrapper expects provider_call(system_prompt, user_input, model,
# temperature, label) -> object with .text. Your call_gemini already returns
# response.text directly, so we wrap it to look object-shaped.
# -----------------------------------------------------------------------------

class _TextWrapper:
    """Minimal stand-in for an SDK response object — only .text is read."""
    __slots__ = ("text",)
    def __init__(self, text: str):
        self.text = text


def gemini_provider_call(*, system_prompt, user_input, model, temperature, label):
    """Adapter: route through the existing call_gemini in this notebook.

    NOTE: existing call_gemini reads MODEL_NAME from globals. We pass `model`
    here for envelope/logging metadata, but the actual model used is whatever
    MODEL_NAME is set to. If you later parameterize call_gemini to take a
    model argument, plumb it through here.
    """
    text = call_gemini(
        system_prompt=system_prompt,
        user_input=user_input,
        temperature=temperature,
        label=label,
    )
    return _TextWrapper(text)


# -----------------------------------------------------------------------------
# Default policy. Tunable in one place.
# -----------------------------------------------------------------------------

DEFAULT_RETRY_POLICY = RetryPolicy(
    max_attempts=3,    # total tries, NOT just retries
    base_delay_s=5.0,  # 503/timeout: ~5s, ~10s, ~20s with jitter
    factor=2.0,
    jitter_frac=0.2,
    quota_buffer_s=3.0,  # added to server's retryDelay on per-minute 429
    max_sleep_s=90.0,    # safety cap on any single sleep
)


# -----------------------------------------------------------------------------
# Public entry point for step functions. One line replaces the bare
# call_gemini(...) call inside extract_filters / decompose_query / generate_code.
#
# Returns the envelope directly. Step functions then layer parse + validation
# on top of a successful infra call.
# -----------------------------------------------------------------------------

def call_gemini_safely(
    system_prompt: str,
    user_input: str,
    *,
    stage: str,           # "step3" | "step4" | "step6"
    label: str | None = None,
    temperature: float = 0.0,
    policy: RetryPolicy | None = None,
) -> dict:
    """Call Gemini with retry + structured failure. Returns the envelope.

    Success:
        {"ok": True, "text": "<response text>", "error": None,
         "stage": stage, "attempts": <int>}

    Failure (infra/quota/client):
        {"ok": False, "text": None,
         "error": {flat dict with stage, kind, status_code, retryable,
                   attempts, retry_after_s, message, model}}

    The step function is responsible for parse + validation on the text and
    for converting parse/validation failures into envelope errors with the
    appropriate `kind`. This function only handles the provider layer.
    """
    try:
        text = call_gemini_with_retry(
            system_prompt, user_input,
            provider_call=gemini_provider_call,
            model=MODEL_NAME,
            temperature=temperature,
            label=label,
            policy=policy or DEFAULT_RETRY_POLICY,
        )
        return {
            "ok": True,
            "text": text,
            "error": None,
            "stage": stage,
            "attempts": 1,  # success path; if you want exact retry-success
                            # counts, plumb that out of the wrapper later
        }
    except GeminiError as e:
        kind = {
            "GeminiInfraError": "infra",
            "GeminiQuotaPerMinuteError": "quota_per_minute",
            "GeminiQuotaDailyError": "quota_daily",
            "GeminiClientError": "client_error",
            "GeminiUnknownError": "unknown",
        }.get(type(e).__name__, "unknown")
        return {
            "ok": False,
            "text": None,
            "error": e.to_envelope_error(stage=stage, kind=kind),
        }


---
## Step 1: Query Normalizer (rule-based)


In [ ]:
# April-25 patch after Phase_3 run:
# delete the two unprefixed patterns. This is one line of code and probably solves more problems than it
#  causes. Worth running your 73-test normalizer suite to confirm no legitimate
#   "films san francisco" → "films" tests regress, but I'd bet none do.

CITY_PATTERNS = [
    re.compile(r'\bin\s+san\s+francisco\b', re.I),
    re.compile(r'\bin\s+sf\b', re.I),
    re.compile(r'\bin\s+the\s+city\b', re.I),
    re.compile(r'\bin\s+california\b', re.I),
    # re.compile(r'\bsan\s+francisco\b', re.I),
    # re.compile(r'\bsf\b', re.I),
]

SYNONYMS = {
    "movie": "film", "movies": "films",
    "acted in": "appeared in", "stars": "actors",
    "shot at": "filmed at", "filmed in": "filmed at",
    "screenwriter": "writer", "screenwriters": "writers",
}

STOP_WORDS = {
    'a','an','the','in','on','at','to','for','of','by','with','and','or','but',
    'is','was','are','were','been','be','have','has','had','do','does','did',
    'will','would','could','should','may','might','can','shall','that','this',
    'these','those','it','its','my','me','i','we','us','our','you','your',
    'he','she','him','her','his','they','them','their','what','which','who',
    'whom','where','when','how','why','if','then','than','so','not','no','nor',
    'from','about','each','every','all','both','few','many','much','more','most',
    'some','any','such','into','over','after','before','between','also','just',
    'only','very','too','here','there',
}

# added April 2026 from Phase 3 normalizer corruption: port, pier --> port, pier, bridge, hotel, ...
# not a real fix
# Real fix — raise the bar for short clusters. The simplest hardening: if a cluster is a single word and
# that word is fewer than ~6 characters, require a higher fuzzy cutoff (say 0.90) before accepting a name
# correction. "port" → "Bud Cort" at ratio ~0.75 would be rejected; genuine typos like "haigh" → "Haigh" stay
# fine because they hit ratio 1.0. This costs you nothing on the legitimate-typo cases and closes the structural hole.
# Even better — gate person-column matches on cluster length. A single-word cluster matching
# against Actor/Director/Writer should only succeed if it's matching the last name of
# someone (which the partial-name path already does) AND the ratio is high. Three-character or four-character
# single-word matches against full names should be auto-rejected. This is essentially: "port"
#  can't become "bud cort" unless port ≈ cort at ratio ≥ ~0.9, which it isn't.

DOMAIN_WORDS = {
    'film','films','movie','movies','show','shows','directed','written','acted',
    'starring','featured','filmed','shot','actor','actors','actress','director',
    'directors','writer','writers','location','locations','scene','scenes','set',
    'list','find','search','count','how','many','compare','top','most','least',
    'best','worst','first','last','near','within','around','mile','miles','km',
    'year','years','decade','during','between','from','appeared','made','produced',
    'released', 'port','pier','hotel','street','avenue','boulevard',
    'cross','bay','beach', 'museum', 'cable', 'car', 'terminal', 'station', 'school', 'hospital', 'building', 'house'
}

FUZZY_CUTOFFS = {'Director':0.75,'Writer':0.75,'Actor':0.75,'Title':0.80,'Locations':0.65}

def is_numeric(word):
    return bool(re.match(r'^\d+s?$', word))

def extract_content_clusters(words):
    clusters, start = [], None
    for i, w in enumerate(words):
        wl = w.lower()
        if wl in STOP_WORDS or wl in DOMAIN_WORDS or is_numeric(wl):
            if start is not None:
                clusters.append((start, i)); start = None
        else:
            if start is None: start = i
    if start is not None: clusters.append((start, len(words)))
    return clusters

# Added is_unjustified_expansion on April-26 after failing Phase_3 test. More info below:
# A genuine fuzzy correction looks like: cluster hitchock (8 chars) → match Hitchcock (9 chars),
# where the match has 1 character of difference and the lengths are similar.
# An unjustified expansion looks like: cluster san francisco (13 chars) → match San Francisco Bay (17 chars),
#  where the cluster is a verbatim prefix and the match adds a whole new word.

def is_unjustified_expansion(cluster_text, matched_value):
    """Reject matches that just append words to a verbatim cluster."""
    cluster_lower = cluster_text.lower().strip()
    matched_lower = matched_value.lower().strip()
    # If the cluster appears verbatim as a prefix or whole word in the match,
    # and the match is strictly longer, this is an expansion, not a correction.
    if cluster_lower == matched_lower:
        return False  # exact match, fine
    if matched_lower.startswith(cluster_lower + ' '):
        return True   # "san francisco" → "san francisco bay" — reject
    if matched_lower.endswith(' ' + cluster_lower):
        return True   # "the bridge" → "golden gate bridge" — reject
    return False


def fuzzy_match_cluster(cluster_text, known_values):
    candidates = [cluster_text]
    if cluster_text.lower().startswith('the '): candidates.append(cluster_text[4:])
    else: candidates.append('the ' + cluster_text)
    for col in ['Locations','Actor','Director','Writer','Title']:
        if col not in known_values: continue
        cutoff = FUZZY_CUTOFFS.get(col, 0.75)
        for cand in candidates:
            matches = difflib.get_close_matches(cand, known_values[col], n=1, cutoff=cutoff)
            if matches:
                ratio = difflib.SequenceMatcher(None, cand.lower(), matches[0].lower()).ratio()
                if is_unjustified_expansion(cand, matches[0]): continue
                if ratio >= cutoff: return (matches[0], col, cluster_text)
            if col in ('Director','Writer','Actor') and ' ' not in cand:
                for full_name in known_values[col]:
                    parts = full_name.split()
                    if len(parts) >= 2:
                        ratio = difflib.SequenceMatcher(None, cand.lower(), parts[-1].lower()).ratio()
                        if ratio >= cutoff: return (full_name, col, cluster_text)
    return None

def normalize_query(query, known_values):
    corrections = []
    q = unicodedata.normalize('NFKC', query).strip().lower()
    q = re.sub(r'\s+', ' ', q)
    for p in CITY_PATTERNS: q = p.sub('', q)
    q = re.sub(r'\s+', ' ', q).strip()
    for old, new in SYNONYMS.items(): q = q.replace(old, new)
    words = q.split()
    clusters = extract_content_clusters(words)
    offset = 0
    for start, end in clusters:
        adj_s, adj_e = start + offset, end + offset
        cluster_text = ' '.join(words[adj_s:adj_e])
        match = fuzzy_match_cluster(cluster_text, known_values)
        if match:
            matched_value, col, original = match
            if matched_value.lower() != cluster_text.lower():
                corrections.append({'original': cluster_text, 'corrected': matched_value, 'column': col})
                new_words = matched_value.lower().split()
                words[adj_s:adj_e] = new_words
                offset += len(new_words) - (end - start)

    # NEW: final canonical pass on suffixes
    normalized = normalize_street_suffixes(' '.join(words)).lower()
    return {'normalized': normalized, 'corrections': corrections}

In [ ]:
# queries_to_test = ['films with roger moore at port of san francisco',
# 'which actors filmed at port of san francisco in 1985',
# 'films called time after time or shot at pier 43',
# 'show filming locations for the oa part ii at pier 43']

# for query in queries_to_test:
#   print(normalize_query(query, known_values))


---
## Step 2: Safety Gate (rule-based)


In [ ]:
SAFE_PHRASES = [
    "add locations to","add location to","remove duplicates","drop duplicates",
    "drop me a list","update me","alter my search","change my search",
    "modify my search","insert a filter","delete filter","remove filter",
]
BLOCKED_PHRASES = [
    "save changes","add a new","add new","change the",
    "export modified","write to database","write to db",
]
BLOCKED_VERBS = [re.compile(rf'\b{v}\b') for v in
    ['delete','update','drop','insert','remove','modify','alter','edit','truncate','overwrite']]

def _find_spans(text, patterns):
    spans = []
    for p in patterns:
        if isinstance(p, re.Pattern):
            for m in p.finditer(text): spans.append((m.start(), m.end()))
        else:
            s = 0
            while True:
                idx = text.find(p, s)
                if idx == -1: break
                spans.append((idx, idx + len(p))); s = idx + 1
    return spans

def _is_covered(span, safe_spans):
    return any(span[0] >= ss and span[1] <= se for ss, se in safe_spans)

def safety_check(query):
    q = query.lower()
    safe_spans = _find_spans(q, SAFE_PHRASES)
    blocked = []
    for phrase in BLOCKED_PHRASES:
        for s, e in _find_spans(q, [phrase]): blocked.append((s, e, phrase))
    for verb_re in BLOCKED_VERBS:
        for m in verb_re.finditer(q): blocked.append((m.start(), m.end(), m.group()))
    for start, end, blocked_by in blocked:
        if not _is_covered((start, end), safe_spans):
            return {'safe': False, 'blocked_by': blocked_by,
                    'message': "This operation cannot be performed as it would modify the database. Only read-only operations are permitted."}
    return {'safe': True, 'blocked_by': None, 'message': None}

---
## Step 3: Task Decomposer (hybrid: short-circuit + LLM)


In [ ]:
MULTI_INTENT_PATTERNS = [
    re.compile(r'\bcompare\b', re.I), re.compile(r'\bhow many\b', re.I),
    re.compile(r'\bwhy\b', re.I), re.compile(r'\band which\b', re.I),
    re.compile(r'\band how\b', re.I), re.compile(r'\band what\b', re.I),
    re.compile(r'\band where\b', re.I), re.compile(r'\band list\b', re.I),
    re.compile(r'\band show\b', re.I), re.compile(r'\band their\b', re.I),
    re.compile(r'\bthen\b', re.I), re.compile(r'\balso\b', re.I),
    re.compile(r'\bas well as\b', re.I), re.compile(r'\bplus\b', re.I),
]

TASK_DECOMPOSER_PROMPT = """You are a query decomposer for a San Francisco film locations database.

Your ONLY job: break a user query into the fewest atomic semantic tasks needed to answer it.

## Output Schema (JSON)
{"tasks": [{"id": "t1", "kind": "retrieve | compare | rank | count | explain | clarify", "source": "exact contiguous text span from the query", "dependsOn": []}]}

## Rules
1. Decomposition ONLY. Do NOT extract filters, mention columns, or reference implementation.
2. Use contiguous spans from the user's query for "source". Do not rephrase.
3. Multi-condition ≠ multi-task. "Films by Hitchcock from the 1950s at North Beach" is ONE retrieve.
4. Max 4 tasks. IDs sequential: t1, t2, t3, t4.
5. dependsOn must reference earlier task IDs only. Omit or use [] if none.
6. Valid kinds: retrieve, compare, rank, count, explain, clarify.
7. No schema words (table, column, sql, join, filter, dataframe, merge) in source.

## Examples
Query: "films directed by hitchcock"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films directed by hitchcock","dependsOn":[]}]}

Query: "what films shot in tenderloin and how many are there"
{"tasks":[{"id":"t1","kind":"retrieve","source":"what films shot in tenderloin","dependsOn":[]},{"id":"t2","kind":"count","source":"how many are there","dependsOn":["t1"]}]}

Query: "compare films in chinatown with ones around north beach"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films in chinatown","dependsOn":[]},{"id":"t2","kind":"retrieve","source":"films around north beach","dependsOn":[]},{"id":"t3","kind":"compare","source":"compare films in chinatown with ones around north beach","dependsOn":["t1","t2"]}]}

Query: "actors who appeared in the most films in 1982"
{"tasks":[{"id":"t1","kind":"rank","source":"actors who appeared in the most films in 1982","dependsOn":[]}]}

Query: "top 5 directors with the most films"
{"tasks":[{"id":"t1","kind":"rank","source":"top 5 directors with the most films","dependsOn":[]}]}

Query: "films within 1 mile radius of coit tower"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films within 1 mile radius of coit tower","dependsOn":[]}]}

Query: "how many films were shot in the 80s"
{"tasks":[{"id":"t1","kind":"count","source":"how many films were shot in the 80s","dependsOn":[]}]}

Query: "films by hitchcock from the 1950s at north beach"
{"tasks":[{"id":"t1","kind":"retrieve","source":"films by hitchcock from the 1950s at north beach","dependsOn":[]}]}

Query: "what is this database about"
{"tasks":[{"id":"t1","kind":"clarify","source":"what is this database about","dependsOn":[]}]}
"""

VALID_KINDS = {'retrieve','compare','rank','count','explain','clarify'}
SCHEMA_LEAK = {'table','column','sql','join','filter','dataframe','merge','index','row','select'}

def validate_task_plan(plan, query):
    if 'tasks' not in plan or not isinstance(plan['tasks'], list): return False
    tasks = plan['tasks']
    if len(tasks) == 0 or len(tasks) > 4: return False
    seen = set()
    for i, t in enumerate(tasks):
        if not all(k in t for k in ('id','kind','source')): return False
        if t['id'] != f"t{i+1}": return False
        seen.add(t['id'])
        if t['kind'] not in VALID_KINDS: return False
        if not t.get('source','').strip(): return False
        if set(t['source'].lower().split()) & SCHEMA_LEAK: return False
        for dep in t.get('dependsOn', []):
            if dep not in seen or dep == t['id']: return False
    return True



def decompose_query(query):
    """Step 3: break the query into atomic semantic tasks.

    Returns the {ok, result, error} envelope. Never returns a fallback
    single-task retrieve on failure — that was the T18/T19 silent-wrong-
    answer bug.
    """
    # --- Deterministic short-circuit: no multi-intent signals → 1 retrieve ---
    # No LLM call, no failure modes. Always succeeds.
    if not any(p.search(query) for p in MULTI_INTENT_PATTERNS):
        return {
            "ok": True,
            "result": {
                "tasks": [
                    {"id": "t1", "kind": "retrieve",
                     "source": query, "dependsOn": []}
                ]
            },
            "error": None,
        }

    # --- Diagnostic: which pattern triggered the LLM path ---
    for p in MULTI_INTENT_PATTERNS:
        if p.search(query):
            print(f"  [decomposer] matched pattern: {p.pattern}")
            break

    # --- Provider call (with retry/backoff/classification) ---
    call = call_gemini_safely(
        TASK_DECOMPOSER_PROMPT,
        query,
        stage="step3",
        label=f"step3_{query}",
    )

    if not call["ok"]:
        # Infra / quota / client / unknown — error already structured.
        err = call["error"]
        print(f"  [decomposer] Provider failure: "
              f"kind={err['kind']} status={err['status_code']} "
              f"attempts={err['attempts']} retryable={err['retryable']}")
        if err.get("retry_after_s"):
            print(f"  [decomposer] Server suggested retry after "
                  f"{err['retry_after_s']}s")
        return {"ok": False, "result": None, "error": err}

    raw_text = call["text"]

    # --- Parse ---
    try:
        plan = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"  [decomposer] JSON parse failure: {e}")
        print(f"  [decomposer] Raw response (first 500 chars): "
              f"{raw_text[:500]}")
        return {
            "ok": False,
            "result": None,
            "error": {
                "stage": "step3",
                "kind": "parse",
                "status_code": None,
                "retryable": False,
                "attempts": 1,
                "retry_after_s": None,
                "message": f"JSONDecodeError: {e}",
                "model": MODEL_NAME,
            },
        }

    # --- Validate against expected schema ---
    if not validate_task_plan(plan, query):
        print("  [decomposer] Validation failed")
        print(f"  [decomposer] Rejected payload: "
              f"{json.dumps(plan, indent=2)[:500]}")
        return {
            "ok": False,
            "result": None,
            "error": {
                "stage": "step3",
                "kind": "validation",
                "status_code": None,
                "retryable": False,
                "attempts": 1,
                "retry_after_s": None,
                "message": "task plan failed schema validation",
                "model": MODEL_NAME,
            },
        }

    # --- Success ---
    return {"ok": True, "result": plan, "error": None}



---
## Step 4: Filter Extractor (LLM) + Post-processors


In [ ]:
FILTER_EXTRACTOR_PROMPT = """You are a filter extractor for a San Francisco film locations database.

## Input Guarantees
The query has already been normalized (lowercase, city terms removed, typos corrected).
You receive a task plan with tasks, each having id, kind, source, and dependsOn.

## Schema
Columns: Title (str), Year (Int64), Locations (str), Fun_Facts (str), Production_Company (str), Distributor (str), Director (str), Writer (str), Actor_1 (str), Actor_2 (str), Actor_3 (str), Neighborhood (str), Supervisor_District (Int64), geometry (Point)
- Actor is a VIRTUAL field meaning "search Actor_1, Actor_2, Actor_3 with OR logic"
- Neighborhood and Supervisor_District are documented but not yet supported as filter fields in v1; do NOT emit predicates against them. See "Allowed Fields" below for the active list.

## Capabilities
The system can: filter rows, count, rank/sort, group by column, compute frequencies.

## MOST IMPORTANT RULE: Constraint vs Output Dimension
A predicate defines what must be TRUE of a row. It does NOT encode what the query asks about.
- "Films by Hitchcock" → Director IS a constraint (predicate: Director == hitchcock)
- "Which directors filmed on Geary St" → Director is the OUTPUT DIMENSION, NOT a constraint. Only Locations is a constraint.
- "Top 5 directors with the most filming locations in 1982" → Only Year is a constraint. Director is the ranking output.

## Predicate Format
Each task gets a `predicate` field: either null or a recursive boolean tree.
Logic node: {"logic": "AND"|"OR", "clauses": [...]}
Leaf clause: {"field": "<col>", "op": "<op>", "value": "<val>", "type": "attribute"|"spatial"}

## Allowed Ops
==            exact match (person, title, single year)
contains      partial text match (Locations, Title, Fun_Facts)
between       inclusive range [low, high] (Year)
> < >= <=     numeric comparison (Year)
within_distance  spatial proximity (geometry field, value: {reference_place, distance, unit})
is_null       field is absent or empty (no value key; only for Director and Writer)
is_not_null   field is present and non-empty (no value key; only for Director and Writer)

## Allowed Fields
Title, Year, Locations, Fun_Facts, Director, Writer, Actor (virtual), geometry

## Field Mapping Rules
1. Person names (director/writer/actor) → use == with lowercase value
2. Location/street/neighborhood names → Locations contains
3. Film titles → Title contains or ==
4. Year/decade/era → Year with ==, between, or comparison ops
5. "near X" / "around X" / "at X" → Locations contains (NOT spatial)
6. "within N miles of X" → geometry within_distance (IS spatial)
7. Fun facts / trivia → Fun_Facts contains
8. Actor/starring → field: "Actor" (virtual, expanded later)
9. Absence language — "no listed director," "without a director," "missing director,"
"unknown director," and the same forms for "writer" — use is_null on the corresponding field.
Omit the value key. v1 supports Director and Writer only — never use is_null on any other field, including Production_Company, Distributor, Neighborhood, or Supervisor_District. For ambiguous phrasings like "films with no X" where X is a person, title, or location, prefer normal == or contains and do NOT use is_null.

## Dependency Rules
1. If a task has dependsOn, read the parent's source text for context but do NOT copy parent predicates.
2. Dependent count/rank tasks with no additional constraints get predicate: null.
3. If a dependent task adds its own constraint (e.g., "why is vertigo..."), extract only that new constraint.

## Task Kind Guidance
- retrieve: extract all row constraints from source
- count: extract constraints if present, null if purely referential
- rank: extract only filtering constraints (year, location), NOT the ranking dimension
- compare: usually null (operates on dependency results)
- explain: extract any specific entity reference (e.g., a film title)
- clarify: always null

## Examples

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films directed by alfred hitchcock"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films directed by alfred hitchcock","predicate":{"field":"Director","op":"==","value":"alfred hitchcock","type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films by alfred hitchcock from the 1950s on sutter st"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films by alfred hitchcock from the 1950s on sutter st","predicate":{"logic":"AND","clauses":[{"field":"Director","op":"==","value":"alfred hitchcock","type":"attribute"},{"field":"Year","op":"between","value":[1950,1959],"type":"attribute"},{"field":"Locations","op":"contains","value":"sutter st","type":"attribute"}]}}]}

Input: {"tasks":[{"id":"t1","kind":"rank","source":"top 5 directors with the most filming locations in 1982"}]}
Output: {"tasks":[{"id":"t1","kind":"rank","source":"top 5 directors with the most filming locations in 1982","predicate":{"field":"Year","op":"==","value":1982,"type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"which directors filmed on geary st"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"which directors filmed on geary st","predicate":{"field":"Locations","op":"contains","value":"geary st","type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films with clint eastwood on geary st"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films with clint eastwood on geary st","predicate":{"logic":"AND","clauses":[{"field":"Actor","op":"==","value":"clint eastwood","type":"attribute"},{"field":"Locations","op":"contains","value":"geary st","type":"attribute"}]}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films within 1 mile of coit tower"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films within 1 mile of coit tower","predicate":{"field":"geometry","op":"within_distance","value":{"reference_place":"coit tower","distance":1,"unit":"mile"},"type":"spatial"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films near golden gate bridge"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films near golden gate bridge","predicate":{"field":"Locations","op":"contains","value":"golden gate bridge","type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films with no listed director"}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films with no listed director","predicate":{"field":"Director","op":"is_null","type":"attribute"}}]}

Input: {"tasks":[{"id":"t1","kind":"retrieve","source":"films shot on valencia st","dependsOn":[]},{"id":"t2","kind":"count","source":"how many are there","dependsOn":["t1"]}]}
Output: {"tasks":[{"id":"t1","kind":"retrieve","source":"films shot on valencia st","predicate":{"field":"Locations","op":"contains","value":"valencia st","type":"attribute"},"dependsOn":[]},{"id":"t2","kind":"count","source":"how many are there","predicate":null,"dependsOn":["t1"]}]}

Input: {"tasks":[{"id":"t1","kind":"rank","source":"actors who appeared in the most films in the 90s"}]}
Output: {"tasks":[{"id":"t1","kind":"rank","source":"actors who appeared in the most films in the 90s","predicate":{"field":"Year","op":"between","value":[1990,1999],"type":"attribute"}}]}

## Strict Rules
1. Only extract ROW CONSTRAINTS. Never encode ranking, grouping, or output shape.
2. Preserve id, kind, source, and dependsOn exactly as received.
3. Values stay lowercase as received from normalization.
4. Do not invent fuller name forms. Use the value as-is.
5. One predicate per task. Use logic nodes for multiple constraints.
6. Null predicate is valid and expected for dependent/compare/clarify tasks.
7. Actor is always virtual — never emit Actor_1/Actor_2/Actor_3 directly.
8. Only use within_distance for explicit distance language.
9. Do not add constraints not stated in the source text.
10. between always uses a 2-element array [low, high].
11. Return the complete tasks array with all original fields preserved.
12. Output valid JSON only.
13. is_null and is_not_null are unary — emit no value key. v1 supports only Director and Writer.
"""

# v1 active filter fields. Neighborhood and Supervisor_District exist in the schema
# but are intentionally excluded here until Phase D enables them as filter fields.
# See: dataset_swap_2026.md, Post_data_swap_list_of_actions.md.
VALID_FIELDS = {'Title','Year','Locations','Fun_Facts','Director','Writer','Actor','geometry'}
VALID_OPS = {'==','contains','between','>','<','>=','<=','within_distance','is_null','is_not_null'}

UNARY_OPS = {'is_null','is_not_null'}
NULL_CAPABLE_FIELDS = {'Director','Writer'}

def _validate_predicate(pred):
    if pred is None: return True
    if 'logic' in pred:
        if pred['logic'] not in ('AND','OR'): return False
        if 'clauses' not in pred: return False
        return all(_validate_predicate(c) for c in pred['clauses'])
    # Leaf clause
    if not all(k in pred for k in ('field','op','type')): return False
    if pred['field'] not in VALID_FIELDS: return False
    if pred['op'] not in VALID_OPS: return False
    if pred['op'] == 'between' and (not isinstance(pred.get('value'), list) or len(pred['value']) != 2): return False
    if pred['op'] == 'within_distance':
        if pred['field'] != 'geometry' or pred['type'] != 'spatial': return False
        v = pred.get('value', {})
        if not all(k in v for k in ('reference_place','distance','unit')): return False

    if pred['op'] in UNARY_OPS:
      if pred['field'] not in NULL_CAPABLE_FIELDS: return False
      if 'value' in pred: return False
    return True

def validate_filter_result(result, input_plan):
    if 'tasks' not in result: return False
    input_tasks = input_plan['tasks']
    output_tasks = result['tasks']
    if len(output_tasks) != len(input_tasks): return False
    for inp, out in zip(input_tasks, output_tasks):
        if out.get('id') != inp['id']: return False
        if out.get('kind') != inp['kind']: return False
        if out.get('source') != inp['source']: return False
        if not _validate_predicate(out.get('predicate')): return False
    return True


### Refactored Extract Filters

In [ ]:

def extract_filters(task_plan, user_query):
    """Step 4: attach a predicate tree to each task in the task plan.

    Returns the {ok, result, error} envelope. Never returns a fallback
    null-predicate IR on failure — that was the silent-wrong-answer bug.
    """
    input_json = json.dumps(task_plan)

    # --- Provider call (with retry/backoff/classification) ---
    call = call_gemini_safely(
        FILTER_EXTRACTOR_PROMPT,
        input_json,
        stage="step4",
        label=f"step4_{user_query[:30]}",
    )

    if not call["ok"]:
        # Infra / quota / client / unknown — error already structured.
        err = call["error"]
        print(f"  [filter_extractor] Provider failure: "
              f"kind={err['kind']} status={err['status_code']} "
              f"attempts={err['attempts']} retryable={err['retryable']}")
        if err.get("retry_after_s"):
            print(f"  [filter_extractor] Server suggested retry after "
                  f"{err['retry_after_s']}s")
        return {"ok": False, "result": None, "error": err}

    raw_text = call["text"]

    # --- Parse ---
    try:
        result = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"  [filter_extractor] JSON parse failure: {e}")
        print(f"  [filter_extractor] Raw response (first 500 chars): "
              f"{raw_text[:500]}")
        return {
            "ok": False,
            "result": None,
            "error": {
                "stage": "step4",
                "kind": "parse",
                "status_code": None,
                "retryable": False,
                "attempts": 1,
                "retry_after_s": None,
                "message": f"JSONDecodeError: {e}",
                "model": MODEL_NAME,
            },
        }

    # --- Normalize envelope: bare-array → wrapped (existing behavior) ---
    if isinstance(result, list):
        print("  [filter_extractor] Normalizing bare-array response")
        result = {"tasks": result}

    # --- Validate against expected schema ---
    if not validate_filter_result(result, task_plan):
        print("  [filter_extractor] Validation failed")
        print(f"  [filter_extractor] Rejected payload: "
              f"{json.dumps(result, indent=2)[:500]}")
        return {
            "ok": False,
            "result": None,
            "error": {
                "stage": "step4",
                "kind": "validation",
                "status_code": None,
                "retryable": False,
                "attempts": 1,
                "retry_after_s": None,
                "message": "filter result failed schema validation",
                "model": MODEL_NAME,
            },
        }

    # --- Success ---
    return {"ok": True, "result": result, "error": None}

### Actors unrolling + Geo resolver post processors

In [ ]:
# === ACTOR EXPANSION POST-PROCESSOR ===
def _expand_actor_in_predicate(pred):
    if pred is None: return None
    if 'logic' in pred:
        pred['clauses'] = [_expand_actor_in_predicate(c) for c in pred['clauses']]
        return pred
    if pred.get('field') == 'Actor':
        value = pred['value']
        op = pred['op']
        ptype = pred['type']
        return {
            "logic": "OR",
            "clauses": [
                {"field": f"Actor_{i}", "op": op, "value": value, "type": ptype}
                for i in range(1, 4)
            ]
        }
    return pred

def expand_actors_in_result(ir):
    for task in ir.get('tasks', []):
        task['predicate'] = _expand_actor_in_predicate(task.get('predicate'))
    return ir


In [ ]:
# === GEOCODING RESOLVER POST-PROCESSOR ===
SF_LANDMARKS = {
    "coit tower": [37.8024, -122.4058],
    "golden gate bridge": [37.8199, -122.4783],
    "alcatraz": [37.8267, -122.4233],
    "alcatraz island": [37.8267, -122.4233],
    "union square": [37.7879, -122.4074],
    "city hall": [37.7793, -122.4193],
    "ferry building": [37.7955, -122.3937],
    "fishermans wharf": [37.8080, -122.4177],
    "fisherman's wharf": [37.8080, -122.4177],
    "pier 39": [37.8087, -122.4098],
    "lombard street": [37.8021, -122.4187],
    "transamerica pyramid": [37.7952, -122.4028],
    "palace of fine arts": [37.8028, -122.4483],
    "twin peaks": [37.7544, -122.4477],
    "dolores park": [37.7596, -122.4269],
    "alamo square": [37.7764, -122.4346],
    "chinatown": [37.7941, -122.4078],
    "north beach": [37.8060, -122.4103],
    "tenderloin": [37.7847, -122.4141],
    "nob hill": [37.7930, -122.4161],
    "russian hill": [37.8011, -122.4194],
    "pacific heights": [37.7925, -122.4352],
    "presidio": [37.7989, -122.4662],
    "the presidio": [37.7989, -122.4662],
    "golden gate park": [37.7694, -122.4862],
    "embarcadero": [37.7936, -122.3930],
    "the embarcadero": [37.7936, -122.3930],
    "treasure island": [37.8235, -122.3707],
    "fort point": [37.8106, -122.4770],
    "fort mason": [37.8060, -122.4310],
    "ghirardelli square": [37.8060, -122.4230],
    "telegraph hill": [37.8027, -122.4057],
    "ocean beach": [37.7600, -122.5100],
    "baker beach": [37.7936, -122.4836],
    "mission district": [37.7599, -122.4148],
    "the mission": [37.7599, -122.4148],
    "castro": [37.7609, -122.4350],
    "the castro": [37.7609, -122.4350],
    "soma": [37.7785, -122.3950],
    "civic center": [37.7796, -122.4157],
    "japantown": [37.7854, -122.4294],
    "oracle park": [37.7786, -122.3893],
    "moscone center": [37.7840, -122.4010],
    "grace cathedral": [37.7919, -122.4130],
    "washington square": [37.8004, -122.4104],
    "yerba buena gardens": [37.7850, -122.4025],
    "market street": [37.7853, -122.4073],
    "cable car turnaround": [37.7847, -122.4079],
    "haight ashbury": [37.7692, -122.4481],
    "haight-ashbury": [37.7692, -122.4481],
    "bernal heights": [37.7430, -122.4153],
    "potrero hill": [37.7600, -122.4002],
    "glen park": [37.7340, -122.4330],
    "cliff house": [37.7787, -122.5136],
}

def _geocode_place(name, landmarks):
    key = name.strip().lower()
    if key in landmarks: return tuple(landmarks[key])
    if key.startswith("the ") and key[4:] in landmarks: return tuple(landmarks[key[4:]])
    if ("the " + key) in landmarks: return tuple(landmarks["the " + key])
    raise ValueError(f"Could not geocode '{name}'. Not in landmarks file.")

def _resolve_geocoding_pred(pred, landmarks):
    if pred is None: return None
    if 'logic' in pred:
        for c in pred['clauses']: _resolve_geocoding_pred(c, landmarks)
        return pred
    if pred.get('type') == 'spatial' and pred.get('op') == 'within_distance':
        val = pred.get('value', {})
        if 'latitude' not in val:
            lat, lon = _geocode_place(val['reference_place'], landmarks)
            val['latitude'] = lat
            val['longitude'] = lon
    return pred

def resolve_geocoding(ir, landmarks=None):
    if landmarks is None: landmarks = SF_LANDMARKS
    for task in ir.get('tasks', []):
        _resolve_geocoding_pred(task.get('predicate'), landmarks)
    return ir


---
## Step 5: Presentation Resolver (rule-based)


In [ ]:
_LOCATION_PATTERNS = [
    re.compile(r'\blocation\b', re.I), re.compile(r'\blocations\b', re.I),
    re.compile(r'\bfilming location\b', re.I), re.compile(r'\bfilming locations\b', re.I),
    re.compile(r'\bfilmed at\b', re.I), re.compile(r'\bshot at\b', re.I),
    re.compile(r'\bwhere\b', re.I), re.compile(r'\bplaces\b', re.I),
    re.compile(r'\bspots\b', re.I), re.compile(r'\bsites\b', re.I),
]
_FILM_PATTERNS = [
    re.compile(r'\bfilm\b', re.I), re.compile(r'\bfilms\b', re.I),
    re.compile(r'\btitle\b', re.I), re.compile(r'\btitles\b', re.I),
]
_METRIC_SUPPRESSORS = [
    re.compile(r'\bmost locations\b', re.I), re.compile(r'\bnumber of locations\b', re.I),
    re.compile(r'\bfewest locations\b', re.I),
    re.compile(r'\bhighest number of locations\b', re.I),
    re.compile(r'\blowest number of locations\b', re.I),
]

def _resolve_granularity(task, all_tasks):
    kind = task.get('kind', '')
    source = task.get('source', '')
    if kind in ('count', 'clarify'):
        return 'scalar'
    # Check for metric suppression
    has_metric = any(p.search(source) for p in _METRIC_SUPPRESSORS)
    has_location = any(p.search(source) for p in _LOCATION_PATTERNS)
    if has_location and not has_metric:
        return 'location'
    # Compare: inherit from deps if they agree
    if kind == 'compare':
        deps = task.get('dependsOn', [])
        dep_granularities = set()
        for t in all_tasks:
            if t['id'] in deps and 'response_granularity' in t:
                dep_granularities.add(t['response_granularity'])
        if len(dep_granularities) == 1:
            return dep_granularities.pop()
    return 'film'

def _resolve_offer_map(task):
    kind = task.get('kind', '')
    if kind in ('count', 'clarify'):
        return False
    return True  # generous policy

def resolve_presentation(ir):
    ir = copy.deepcopy(ir)
    tasks = ir.get('tasks', [])
    # Two passes: first resolve non-compare, then compare (needs deps resolved)
    for task in tasks:
        if task.get('kind') != 'compare':
            task['response_granularity'] = _resolve_granularity(task, tasks)
            task['offer_map'] = _resolve_offer_map(task)
    for task in tasks:
        if task.get('kind') == 'compare':
            task['response_granularity'] = _resolve_granularity(task, tasks)
            task['offer_map'] = _resolve_offer_map(task)
    ir['offer_map'] = any(t.get('offer_map', False) for t in tasks)
    return ir


---
## Pipeline Runner (Steps 1–5)


In [ ]:
def run_preprocessing_pipeline(user_query, known_values, verbose=True):
    """Run Steps 1-5 and return the complete IR.

    Returns either:
      - the IR dict (success), OR
      - {'error': True, 'message': str, 'failed_stage': str,
         'stage_error': <flat error envelope>} on a structured failure.

    The latter shape is recognized by run_full_pipeline's existing
    error-check, so the orchestrator short-circuits cleanly.
    """
    if verbose:
        print(f"\n{'='*60}\nQuery: {user_query}\n{'='*60}")

    # --- Step 1: Normalize ---
    norm = normalize_query(user_query, known_values)
    cleaned = norm['normalized']
    if verbose:
        print(f"\n[Step 1] Normalized: {cleaned}")
        if norm['corrections']:
            print(f"  Corrections: {norm['corrections']}")

    # --- Step 2: Safety Gate ---
    gate = safety_check(cleaned)
    if not gate['safe']:
        if verbose:
            print(f"\n[Step 2] BLOCKED: {gate['blocked_by']}")
        return {
            'error': True,
            'message': gate['message'],
            'blocked_by': gate['blocked_by'],
            'failed_stage': 'step2',
        }
    if verbose:
        print(f"[Step 2] Safe ✓")

    # --- Step 3: Task Decomposer (NEW: envelope-aware) ---
    if verbose:
        print(f"\n[Step 3] Decomposing...")

    decompose_envelope = decompose_query(cleaned)

    if not decompose_envelope["ok"]:
        # Honest failure — short-circuit. Do NOT proceed to Step 4, Step 5,
        # codegen, or execution. This is the fix for the T18/T19 silent-
        # wrong-answer class: a 503 in Step 3 used to silently fall back
        # to a single-task retrieve, dropping the count intent. Now it
        # surfaces.
        err = decompose_envelope["error"]
        msg = (f"Step 3 failed: {err['kind']} "
               f"(status={err['status_code']}, attempts={err['attempts']})")
        if verbose:
            print(f"\n❌ {msg}")
            if err.get("retry_after_s"):
                print(f"   Server suggested retry after "
                      f"~{err['retry_after_s']}s")
        return {
            'error': True,
            'message': msg,
            'failed_stage': 'step3',
            'stage_error': err,   # full flat envelope, for the test harness
        }

    task_plan = decompose_envelope["result"]
    if verbose:
        print(f"  Tasks: {json.dumps(task_plan, indent=2)}")
    time.sleep(LLM_DELAY)

    # --- Step 4: Filter Extractor (envelope-aware, unchanged from prior refactor) ---
    if verbose:
        print(f"\n[Step 4] Extracting filters...")

    filter_envelope = extract_filters(task_plan, user_query)

    if not filter_envelope["ok"]:
        err = filter_envelope["error"]
        msg = (f"Step 4 failed: {err['kind']} "
               f"(status={err['status_code']}, attempts={err['attempts']})")
        if verbose:
            print(f"\n❌ {msg}")
            if err.get("retry_after_s"):
                print(f"   Server suggested retry after "
                      f"~{err['retry_after_s']}s")
        return {
            'error': True,
            'message': msg,
            'failed_stage': 'step4',
            'stage_error': err,
        }

    # Unwrap on success and run post-processors as before.
    filter_result = filter_envelope["result"]
    filter_result = expand_actors_in_result(filter_result)
    filter_result = resolve_geocoding(filter_result)
    if verbose:
        print(f"  Filters: {json.dumps(filter_result, indent=2)}")
    time.sleep(LLM_DELAY)

    # --- Step 5: Presentation Resolver ---
    ir = resolve_presentation(filter_result)
    if verbose:
        print(f"\n[Step 5] Presentation resolved:")
        for t in ir['tasks']:
            print(f"  {t['id']}: granularity={t.get('response_granularity')}, "
                  f"offer_map={t.get('offer_map')}")
        print(f"  Top-level offer_map: {ir.get('offer_map')}")

    return ir

---
## Step 6: Code Generation (LLM)


In [ ]:
# === LOAD CODE GENERATION PROMPT ===
# Upload code_generation_v2.md to Colab, then load it here.
# drive.mount('/content/drive')
CODE_GEN_PROMPT_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/code_generation_v2.3.md'

def load_codegen_prompt(path=CODE_GEN_PROMPT_PATH):
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

CODEGEN_PROMPT_TEMPLATE = load_codegen_prompt()
print(f"Loaded code generation prompt: {len(CODEGEN_PROMPT_TEMPLATE)} chars, "
      f"{CODEGEN_PROMPT_TEMPLATE.count(chr(10))} lines")

# Verify injection point exists
assert '{ir_json}' in CODEGEN_PROMPT_TEMPLATE, "Prompt must contain {ir_json} placeholder"
print("✓ {ir_json} placeholder found")


Loaded code generation prompt: 57303 chars, 1619 lines
✓ {ir_json} placeholder found


In [ ]:
def generate_code(ir, user_query, verbose=True):
    """Step 6: Generate executable Python code from the IR.

    Injects the IR into the code generation prompt and calls the LLM via
    the resilience layer. Returns the {ok, result, error} envelope.

    On success, result is {"code": str, "explanation": str}.
    On failure, error is the flat structured envelope.
    """
    ir_json = json.dumps(ir, indent=2)
    prompt = CODEGEN_PROMPT_TEMPLATE.replace('{ir_json}', ir_json)

    if verbose:
        print(f"\n[Step 6] Generating code...")
        print(f"  IR size: {len(ir_json)} chars")
        print(f"  Prompt size: {len(prompt)} chars")

    # --- Provider call (with retry/backoff/classification) ---
    call = call_gemini_safely(
        prompt,
        user_query,
        stage="step6",
        label=f"step6_{user_query[:30]}",
    )

    if not call["ok"]:
        # Infra / quota / client / unknown — error already structured.
        err = call["error"]
        print(f"  [generate_code] Provider failure: "
              f"kind={err['kind']} status={err['status_code']} "
              f"attempts={err['attempts']} retryable={err['retryable']}")
        if err.get("retry_after_s"):
            print(f"  [generate_code] Server suggested retry after "
                  f"{err['retry_after_s']}s")
        return {"ok": False, "result": None, "error": err}

    response_text = call["text"]

    # --- Parse JSON (with regex-extraction fallback for fenced/preambled responses) ---
    parsed = None
    try:
        parsed = json.loads(response_text)
    except json.JSONDecodeError:
        # Try to extract a JSON object from the response. Models occasionally
        # wrap the JSON in ```json ... ``` fences or add a preamble; this
        # recovers from that. NOT a silent success-fallback — if extraction
        # also fails, we surface a parse error.
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group())
            except json.JSONDecodeError as e2:
                print(f"  [generate_code] JSON parse failure (after extraction attempt): {e2}")
                print(f"  [generate_code] Raw response (first 500 chars): {response_text[:500]}")
                return {
                    "ok": False,
                    "result": None,
                    "error": {
                        "stage": "step6",
                        "kind": "parse",
                        "status_code": None,
                        "retryable": False,
                        "attempts": 1,
                        "retry_after_s": None,
                        "message": f"JSONDecodeError after extraction: {e2}",
                        "model": MODEL_NAME,
                    },
                }
        else:
            print(f"  [generate_code] Could not parse or extract JSON from response")
            print(f"  [generate_code] Raw response (first 500 chars): {response_text[:500]}")
            return {
                "ok": False,
                "result": None,
                "error": {
                    "stage": "step6",
                    "kind": "parse",
                    "status_code": None,
                    "retryable": False,
                    "attempts": 1,
                    "retry_after_s": None,
                    "message": "no JSON object found in response",
                    "model": MODEL_NAME,
                },
            }

    # --- Validate: code field must exist and be non-empty ---
    code = parsed.get("code", "")
    explanation = parsed.get("explanation", "")

    if not code or not code.strip():
        print(f"  [generate_code] Validation failed: empty `code` field")
        print(f"  [generate_code] Raw response (first 500 chars): {response_text[:500]}")
        return {
            "ok": False,
            "result": None,
            "error": {
                "stage": "step6",
                "kind": "validation",
                "status_code": None,
                "retryable": False,
                "attempts": 1,
                "retry_after_s": None,
                "message": "LLM returned empty `code` field",
                "model": MODEL_NAME,
            },
        }

    if verbose:
        print(f"  ✓ Code generated: {len(code)} chars")
        print(f"  Explanation: {explanation[:150]}...")

    # --- Success ---
    return {
        "ok": True,
        "result": {"code": code, "explanation": explanation},
        "error": None,
    }

---
## Code Executor


In [ ]:
import ast
import os
import json
from datetime import datetime

EXEC_LOG_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/code_gen_log.txt'

def _append_exec_log(entry: str):
    """Append a formatted entry to the execution log. Silent on failure."""
    try:
        os.makedirs(os.path.dirname(EXEC_LOG_PATH), exist_ok=True)
        with open(EXEC_LOG_PATH, 'a', encoding='utf-8') as f:
            f.write(entry)
            f.write('\n' + ('=' * 72) + '\n\n')
    except Exception as e:
        # Logging must never break execution. Print once so it's visible.
        print(f"  [exec_log] write failed: {e}")


def _format_exec_log_entry(code: str, outcome: dict, query: str = None) -> str:
    """Build a single log entry string from the execution outcome."""
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    lines = [f"[{ts}] code_execution"]
    if query:
        lines.append(f"Query: {query}")
    lines.append(f"Success: {outcome.get('success')}")

    # Unwrap the success-path nesting so the rest of the formatter
    # sees a flat view regardless of which branch produced the outcome.
    if outcome.get('success') and 'result' in outcome and isinstance(outcome['result'], dict):
        payload = outcome['result']
    else:
        payload = outcome

    summary = payload.get('summary')
    if summary:
        lines.append(f"Summary: {summary}")

    meta = payload.get('metadata') or {}
    if meta.get('error'):
        lines.append(f"Error: {meta.get('error_type', 'Error')}: {meta['error']}")
    if meta.get('errors'):
        lines.append(f"Validation errors: {meta['errors']}")
    if meta:
        # Surface other metadata fields (query_type, task_count, task_ids, etc.)
        other_meta = {k: v for k, v in meta.items() if k not in ('error', 'error_type', 'errors')}
        if other_meta:
            lines.append(f"Metadata: {other_meta}")

    # Serialize the data payload. Truncate aggressively — the log is for
    # scanning, not for archiving full result sets.
    data = payload.get('data')
    if data is not None:
        lines.append("")
        lines.append("--- Result data ---")
        try:
            data_str = json.dumps(data, default=str, indent=2)
        except Exception:
            data_str = str(data)
        if len(data_str) > 2000:
            data_str = data_str[:2000] + f"\n... [truncated, {len(data_str)} chars total]"
        lines.append(data_str)

    lines.append("")
    lines.append("--- Generated code ---")
    lines.append(code if code else "(no code)")
    return '\n'.join(lines)


def validate_code(code):
    """Basic validation before execution."""
    errors = []
    try:
        ast.parse(code)
    except SyntaxError as e:
        errors.append(f"Syntax error: {e}")
        return errors

    if 'def process_sf_film_query' not in code:
        errors.append("Missing process_sf_film_query function")

    if 'gdf =' in code and 'gdf_copy' not in code.split('gdf =')[0]:
        errors.append("Possible mutation of input gdf")

    return errors


def execute_code(code, gdf, verbose=True, query=None):
    """Execute generated code against the GeoDataFrame."""
    # Validate first
    validation_errors = validate_code(code)
    if validation_errors:
        outcome = {
            'success': False,
            'data': None,
            'summary': f"Validation failed: {'; '.join(validation_errors)}",
            'metadata': {'errors': validation_errors}
        }
        _append_exec_log(_format_exec_log_entry(code, outcome, query))
        return outcome

    # Execute
    try:
        exec_globals = {}
        exec(code, exec_globals)

        if 'process_sf_film_query' not in exec_globals:
            outcome = {
                'success': False,
                'data': None,
                'summary': 'Function process_sf_film_query not found after execution',
                'metadata': {}
            }
            _append_exec_log(_format_exec_log_entry(code, outcome, query))
            return outcome

        func = exec_globals['process_sf_film_query']
        result = func(gdf)

        # Detect errors the generated code caught internally.
        internal_error = (
            isinstance(result, dict)
            and 'error' in result.get('metadata', {})
        )

        if internal_error:
            if verbose:
                err = result['metadata'].get('error', 'unknown')
                err_type = result['metadata'].get('error_type', 'Error')
                print(f"\n[Execution] ✗ Internal error: {err_type}: {err}")
            outcome = {
                'success': False,
                'data': result.get('data'),
                'summary': result.get('summary', 'Internal execution error'),
                'metadata': result.get('metadata', {})
            }
            _append_exec_log(_format_exec_log_entry(code, outcome, query))
            return outcome

        if verbose:
            print(f"\n[Execution] ✓ Success")
            if isinstance(result, dict):
                print(f"  Summary: {result.get('summary', 'N/A')}")
                data = result.get('data')
                if isinstance(data, (pd.DataFrame, gpd.GeoDataFrame)):
                    print(f"  Data: DataFrame with {len(data)} rows")
                elif isinstance(data, dict):
                    print(f"  Data keys: {list(data.keys())[:10]}")
                elif data is not None:
                    print(f"  Data: {str(data)[:200]}")
            else:
                print(f"  Result: {str(result)[:200]}")

        outcome = {'success': True, 'result': result}
        _append_exec_log(_format_exec_log_entry(code, outcome, query))
        return outcome

    except Exception as e:
        if verbose:
            print(f"\n[Execution] ✗ Error: {type(e).__name__}: {e}")
        outcome = {
            'success': False,
            'data': None,
            'summary': f"Execution error: {str(e)}",
            'metadata': {'error': str(e), 'error_type': type(e).__name__}
        }
        _append_exec_log(_format_exec_log_entry(code, outcome, query))
        return outcome


---
## End-to-End Runner


In [ ]:
def run_full_pipeline(user_query, gdf, known_values, verbose=True):
    """Complete pipeline: preprocessing (Steps 1-5) → code generation → execution.
    Returns all intermediate results for inspection.
    """
    pipeline_result = {
        'query': user_query,
        'ir': None,
        'codegen': None,
        'execution': None,
        'success': False,
    }

    # --- Steps 1-5: Preprocessing ---
    ir = run_preprocessing_pipeline(user_query, known_values, verbose=verbose)

    # --- Check for blocked queries (safety gate) and structured stage failures ---
    if isinstance(ir, dict) and ir.get('error'):
        pipeline_result['ir'] = ir
        if verbose:
            stage = ir.get('failed_stage', 'unknown')
            if stage == 'step2':
                print(f"\n❌ Pipeline stopped (safety gate): {ir.get('message')}")
            elif stage == 'step3':
                err = ir.get('stage_error', {})
                kind = err.get('kind', 'unknown')
                print(f"\n❌ Pipeline stopped (Step 3 {kind}): "
                      f"{ir.get('message')}")
                _print_kind_hint(kind, err)
            elif stage == 'step4':
                err = ir.get('stage_error', {})
                kind = err.get('kind', 'unknown')
                print(f"\n❌ Pipeline stopped (Step 4 {kind}): "
                      f"{ir.get('message')}")
                _print_kind_hint(kind, err)
            else:
                print(f"\n❌ Pipeline stopped: {ir.get('message')}")
        return pipeline_result

    # --- Check for clarify-only tasks ---
    if all(t['kind'] == 'clarify' for t in ir.get('tasks', [])):
        pipeline_result['ir'] = ir
        if verbose:
            print(f"\n⚠️ Clarification needed — skipping code generation")
        return pipeline_result

    pipeline_result['ir'] = ir
    time.sleep(LLM_DELAY)

    # --- Step 6: Code Generation (NEW: envelope-aware) ---
    codegen_envelope = generate_code(ir, user_query, verbose=verbose)

    if not codegen_envelope["ok"]:
        # Honest failure — structured envelope. Same {error, failed_stage,
        # stage_error, message} shape used by Steps 2, 3, 4. We store it on
        # pipeline_result['ir'] so the existing test harness summary code
        # (which keys off ir.get('failed_stage')) catches step6 uniformly.
        err = codegen_envelope["error"]
        msg = (f"Step 6 failed: {err['kind']} "
               f"(status={err['status_code']}, attempts={err['attempts']})")
        pipeline_result['ir'] = {
            'error': True,
            'message': msg,
            'failed_stage': 'step6',
            'stage_error': err,
            # Preserve the upstream IR too — useful for post-mortem on a
            # codegen failure. The test harness ignores this key; it only
            # reads failed_stage / stage_error.
            'preprocessing_ir': ir,
        }
        if verbose:
            print(f"\n❌ Pipeline stopped (Step 6 {err['kind']}): {msg}")
            _print_kind_hint(err['kind'], err)
        return pipeline_result

    # Unwrap on success — pipeline_result['codegen'] gets the same
    # {code, explanation} dict the rest of the notebook (e.g., the
    # "view generated code" cells) expects.
    codegen = codegen_envelope["result"]
    pipeline_result['codegen'] = codegen

    # --- Execution ---
    execution = execute_code(codegen['code'], gdf, verbose=verbose)
    pipeline_result['execution'] = execution
    pipeline_result['success'] = execution.get('success', False)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Pipeline {'✓ SUCCESS' if pipeline_result['success'] else '✗ FAILED'}")
        print(f"{'='*60}")

    return pipeline_result


def _print_kind_hint(kind, err):
    """Friendly user-facing hint based on the error kind. Shared across stages."""
    if kind == 'quota_per_minute' and err.get('retry_after_s'):
        print(f"   → Try again in about "
              f"{int(err['retry_after_s']) + 5} seconds.")
    elif kind == 'quota_daily':
        print(f"   → Daily quota exhausted. Try again tomorrow "
              f"or upgrade the API plan.")
    elif kind == 'infra':
        print(f"   → The model service is temporarily unavailable. "
              f"Try again shortly.")
    # parse / validation / client_error / unknown: no special hint —
    # the structured printout above already says what happened.


---
## Phase 1 (Single-Task Queries)


In [ ]:
PHASE_1_TESTS = [
    {
        "name": "T1: Retrieve by Director (string ==)",
        "query": "films directed by alfred hitchcock",
        "expect": "retrieve, film granularity, Director predicate",
    },
    {
        "name": "T2: Retrieve by Location (contains)",
        "query": "films shot on market street",
        "expect": "retrieve, film granularity, Locations contains",
    },
    {
        "name": "T3: Retrieve by Actor (virtual field)",
        "query": "films with sean penn",
        "expect": "retrieve, film granularity, Actor OR expansion",
    },
    {
        "name": "T4: Count films in a decade",
        "query": "how many films were shot in the 80s",
        "expect": "count, scalar granularity, Year between 1980-1989",
    },
    {
        "name": "T5: Rank directors by film count",
        "query": "top 5 directors with the most films",
        "expect": "rank, film granularity, no predicate filter (null or all rows)",
    },
    {
        "name": "T6: Retrieve by Year (exact)",
        "query": "films from 1985",
        "expect": "retrieve, film granularity, Year == 1985",
    },
    {
        "name": "T7: Retrieve by Director + Year range (AND)",
        "query": "films by alfred hitchcock from the 1950s",
        "expect": "retrieve, film granularity, Director AND Year between",
    },
    {
        "name": "T8: Spatial retrieve (within_distance)",
        "query": "films within 1 mile of coit tower",
        "expect": "retrieve, location granularity, geometry within_distance",
    },
    {
        "name": "T9: Retrieve locations for a film",
        "query": "show filming locations for vertigo",
        "expect": "retrieve, location granularity, Title contains vertigo",
    },
    {
        "name": "T10: Rank actors by film count in decade",
        "query": "actors who appeared in the most films in the 90s",
        "expect": "rank, film granularity, Year between 1990-1999",
    },
]


## Phase-2 - T11-T20 - single queries

In [ ]:
PHASE_2_TESTS = [
    {
        "name": "T11: Disjunction across same field",
        "query": "films by zachary shedd or nicholas meyer",
        "expect": "retrieve, film granularity, Director OR Director",
    },
    {
        "name": "T12: Exact title match",
        "query": "the film called time after time",
        "expect": "retrieve, film granularity, Title exact-match semantics",
    },
    {
        "name": "T13: Title substring retrieval",
        "query": "films with boys in the title",
        "expect": "retrieve, film granularity, Title contains 'boys'",
    },
    {
        "name": "T14: Multi-value director cell",
        "query": "films directed by michel brezis",
        "expect": "retrieve, film granularity, Director match must succeed on multi-director field",
    },
    {
        "name": "T15: Location abbreviation variance",
        "query": "films on larkin street",
        "expect": "retrieve, film granularity, Locations contains normalized larkin st variants",
    },
    {
        "name": "T16: Null / missing-field predicate",
        "query": "films with no listed director",
        "expect": "retrieve, film granularity, Director is null or empty",
    },
    {
        "name": "T17: Disjunction across different fields",
        "query": "films either directed by nicholas meyer or shot at coit tower",
        "expect": "retrieve, film granularity, OR tree across Director and Locations",
    },
    {
        "name": "T18: Retrieve then count dependency",
        "query": "films on larkin street and how many are there",
        "expect": "retrieve plus count, dependency chain, t2 dependsOn t1, count distinct films from t1 matched_rows",
    },
    {
        "name": "T19: Retrieve then location-count dependency",
        "query": "show filming locations for time after time and how many locations are there",
        "expect": "retrieve plus count, dependency chain, location granularity retrieve then count matched rows",
    },
    {
        "name": "T20: Cross-field AND with mixed predicates",
        "query": "films by nicholas meyer at the hyatt regency hotel",
        "expect": "retrieve, film granularity, Director AND Locations contains",
    },
]

## Test Suite

In [ ]:
# =============================================================================
# CELL: Refactored run_test_suite (summary printout adds step3 and step6 branches)
# WHAT CHANGES:
#   - The per-test failure summary now branches on failed_stage for step3,
#     step4, AND step6 — each printing kind/status/attempts uniformly.
#   - The legacy `elif codegen.get('error'):` branch is removed because
#     after the Step 6 refactor, codegen is None on failure (the structured
#     failure lives on ir['stage_error'] instead).
#
# WHAT DOES NOT CHANGE:
#   - The pause logic, the per-test loop, the pass/fail counting.
#   - The execution-error fallback branch (still useful for codegen-success-
#     but-execution-failure cases — those are real and worth surfacing).
# =============================================================================

def run_test_suite(tests, gdf, known_values, pause=2):
    """Run a suite of test queries through the full pipeline."""
    results = []
    total = len(tests)

    for i, test in enumerate(tests):
        print(f"\n{'#'*60}")
        print(f"  TEST {i+1}/{total}: {test['name']}")
        print(f"  Query: {test['query']}")
        print(f"  Expected: {test['expect']}")
        print(f"{'#'*60}")

        result = run_full_pipeline(test['query'], gdf, known_values, verbose=True)
        result['test_name'] = test['name']
        result['expected'] = test['expect']
        results.append(result)

        if i < total - 1:
            print(f"\n  ⏳ Waiting {pause}s before next test...")
            time.sleep(pause)

    # --- Summary ---
    print(f"\n\n{'='*60}")
    print(f"TEST SUITE SUMMARY")
    print(f"{'='*60}")
    passed = sum(1 for r in results if r['success'])
    failed = total - passed
    print(f"  Passed: {passed}/{total}")
    print(f"  Failed: {failed}/{total}")
    print()
    for r in results:
        status = "✓" if r['success'] else "✗"
        print(f"  {status} {r['test_name']}")
        if not r['success']:
            ir = r.get('ir') or {}
            execution = r.get('execution') or {}
            stage = ir.get('failed_stage')
            if stage in ('step3', 'step4', 'step6'):
                err = ir.get('stage_error', {})
                stage_label = {'step3': 'Step 3', 'step4': 'Step 4', 'step6': 'Step 6'}[stage]
                print(f"    → {stage_label} {err.get('kind','?')} "
                      f"(status={err.get('status_code')}, "
                      f"attempts={err.get('attempts')})")
            elif stage == 'step2':
                print(f"    → Safety gate: {ir.get('blocked_by','?')}")
            elif execution:
                print(f"    → Execution error: {execution.get('summary','')[:100]}")
            else:
                print(f"    → Unknown failure: {ir.get('message','(no message)')[:100]}")

    return results


## Phase-3 - T21-T34


In [ ]:
PHASE_3_TESTS = [
    {
        "name": "T21: Cross-field AND (Director + location)",
        "query": "films directed by nicholas meyer at golden gate bridge",
        "expect": "single-task retrieve; film granularity; Director == nicholas meyer AND Locations contains golden gate bridge",
    },
    {
        "name": "T22: Cross-field AND (Actor + location)",
        "query": "films with roger moore at port of san francisco",
        "expect": "single-task retrieve; film granularity; Actor == roger moore AND Locations contains port of san francisco",
    },
    {
        "name": "T23: Cross-field AND (Writer + location)",
        "query": "films written by linda demetrick at hyatt hotel",
        "expect": "single-task retrieve; film granularity; Writer == linda demetrick AND Locations contains hyatt hotel",
    },
    {
        "name": "T24: Cross-field OR (Director OR writer)",
        "query": "films directed by andrew haigh or written by kay cannon",
        "expect": "single-task retrieve; film granularity; OR predicate across Director == andrew haigh and Writer == kay cannon",
    },
    {
        "name": "T25: Cross-field OR (Title OR location)",
        "query": "films called time after time or shot at pier 43",
        "expect": "single-task retrieve; film granularity; OR predicate across Title contains time after time and Locations contains pier 43",
    },
    {
        "name": "T26: Constraint vs output dimension (writers at location)",
        "query": "which writers filmed at golden gate bridge",
        "expect": "single-task retrieve; writer is output dimension, not filter; predicate should be Locations contains golden gate bridge only",
    },
    {
        "name": "T27: Constraint vs output dimension (actors at location and year)",
        "query": "which actors filmed at port of san francisco in 1985",
        "expect": "single-task retrieve; actor is output dimension, not filter; predicate should be Locations contains port of san francisco AND Year == 1985",
    },
    {
        "name": "T28: Cross-field AND (Title + location)",
        "query": "show filming locations for the oa part ii at pier 43",
        "expect": "single-task retrieve; location granularity; Title contains the oa part ii AND Locations contains pier 43",
    },
    {
        "name": "T29: Retrieve -> count (film basis)",
        "query": "films shot at golden gate bridge and how many are there",
        "expect": "two tasks; t1 retrieve with Locations contains golden gate bridge; t2 count dependsOn t1 with null predicate; scalar count_basis should default to films",
    },
    {
        "name": "T30: Retrieve -> count (location basis)",
        "query": "show filming locations for the oa part ii and how many locations are there",
        "expect": "two tasks; t1 retrieve with Title contains the oa part ii and location granularity; t2 count dependsOn t1 with null predicate; scalar count_basis should be locations",
    },
    {
        "name": "T31: Retrieve -> dependent narrow with added predicate",
        "query": "films shot at golden gate bridge and which of those were directed by nicholas meyer",
        "expect": "two tasks; t1 retrieve with Locations contains golden gate bridge; t2 retrieve dependsOn t1 with new predicate Director == nicholas meyer only",
    },
    {
        "name": "T32: Retrieve -> rank within dependency subset",
        "query": "films shot at golden gate bridge and which directors appear most often there",
        "expect": "two tasks; t1 retrieve with Locations contains golden gate bridge; t2 rank dependsOn t1 with null predicate; ranking dimension is Director, not a filter",
    },
    {
        "name": "T33: Retrieve -> retrieve -> compare",
        "query": "films shot at golden gate bridge and films shot at port of san francisco and compare them",
        "expect": "three tasks; t1 retrieve with Locations contains golden gate bridge; t2 retrieve with Locations contains port of san francisco; t3 compare dependsOn [t1, t2] with null predicate",
    },
    {
    "name": "T34: Retrieve -> count with potentially competing film/location cues",
    "query": "show filming locations for time after time and how many films are represented there",
    "expect": "two tasks; t1 retrieve with Title contains time after time and location granularity; t2 count dependsOn t1 with null predicate; primary diagnostic: whether count_basis is decided from t2 source text alone or inherited from t1's location-oriented framing; expected-preferred outcome is films, but locations would be an informative semantic finding rather than a pure regression",
    }
]

PHASE_3_TESTS_A = PHASE_3_TESTS[:8]
PHASE_3_TESTS_B = PHASE_3_TESTS[8:]


---
## Run Tests



### PHASE_3_TESTS_A

In [ ]:
# PHASE_3_TESTS_A
test_results = run_test_suite(PHASE_3_TESTS_A, gdf, known_values)



############################################################
  TEST 1/8: T21: Cross-field AND (Director + location)
  Query: films directed by nicholas meyer at golden gate bridge
  Expected: single-task retrieve; film granularity; Director == nicholas meyer AND Locations contains golden gate bridge
############################################################

Query: films directed by nicholas meyer at golden gate bridge

[Step 1] Normalized: films directed by nicholas meyer at golden gate bridge
[Step 2] Safe ✓

[Step 3] Decomposing...
  Tasks: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films directed by nicholas meyer at golden gate bridge",
      "dependsOn": []
    }
  ]
}

[Step 4] Extracting filters...
Successfully saved to /content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/raw_responses/20260426_074744_step4_films_directed_by_nicholas_mey.json
  Filters: {
  "tasks": [
    {
      "id": "t1",
      "kind"

### PHASE_3_TESTS_B

In [ ]:
test_results = run_test_suite(PHASE_3_TESTS_B, gdf, known_values)


############################################################
  TEST 1/6: T29: Retrieve -> count (film basis)
  Query: films shot at golden gate bridge and how many are there
  Expected: two tasks; t1 retrieve with Locations contains golden gate bridge; t2 count dependsOn t1 with null predicate; scalar count_basis should default to films
############################################################

Query: films shot at golden gate bridge and how many are there

[Step 1] Normalized: films filmed at golden gate bridge and how many are there
[Step 2] Safe ✓

[Step 3] Decomposing...
  [decomposer] matched pattern: \bhow many\b
Successfully saved to /content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/raw_responses/20260426_075710_step3_films_filmed_at_golden_gate_bridge_and_how_many_are_there.json
  Tasks: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films filmed at golden gate bridge",
      "dependsOn": []
    },
    {

In [ ]:
# --- Run a single query first to smoke test ---
# result = run_full_pipeline("films with no listed director", gdf, known_values)
result = run_full_pipeline("films with no listed director", gdf, known_values)


Query: films with no listed director

[Step 1] Normalized: films with no listed director
[Step 2] Safe ✓

[Step 3] Decomposing...
  Tasks: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films with no listed director",
      "dependsOn": []
    }
  ]
}

[Step 4] Extracting filters...
Successfully saved to /content/drive/MyDrive/Colab Notebooks/SF Film Project/calude-refactoring/e2e-testing/raw_responses/20260424_202854_step4_films_with_no_listed_director.json
  Filters: {
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films with no listed director",
      "dependsOn": [],
      "predicate": {
        "field": "Director",
        "op": "is_null",
        "type": "attribute"
      }
    }
  ]
}

[Step 5] Presentation resolved:
  t1: granularity=film, offer_map=True
  Top-level offer_map: True

[Step 6] Generating code...
  IR size: 343 chars
  Prompt size: 57637 chars
Successfully saved to /content/drive/MyDrive/Colab No

In [ ]:
# --- Inspect the generated code ---
if result.get('codegen') and not result['codegen'].get('error'):
    print(result['codegen']['code'])


In [ ]:
# --- Run the full Phase 1 test suite ---
test_results = run_test_suite(PHASE_1_TESTS, gdf, known_values)


In [ ]:
import inspect
src = inspect.getsource(extract_filters)
print(src[:400])
print("---")
print("uses envelope?", '"ok"' in src or "'ok'" in src)

In [ ]:
    # --- Run the full Phase 2 T11-T20 test suite ---
test_results = run_test_suite(PHASE_2_TESTS, gdf, known_values)

---
## Inspect Individual Results

After running tests, inspect specific results:


In [ ]:
# --- View IR for a specific test ---
idx = 1  # change to inspect different test
print(json.dumps(test_results[idx]['ir'], indent=2))


{
  "tasks": [
    {
      "id": "t1",
      "kind": "retrieve",
      "source": "films shot on market st",
      "dependsOn": [],
      "predicate": {
        "field": "Locations",
        "op": "contains",
        "value": "market st",
        "type": "attribute"
      },
      "response_granularity": "film",
      "offer_map": true
    }
  ],
  "offer_map": true
}


In [ ]:
import json

output_data = []

for i in range(10): # For T1 to T10
    test_name_prefix = f"T{i+11}"
    # Find the corresponding test result from PHASE_1_TESTS using test_name
    # The test_results list is ordered the same as PHASE_1_TESTS
    test_idx = i

    if test_idx < len(test_results):
        result = test_results[test_idx]
        generated_code = "NO_GEN_CODE_AVAILABLE"
        if result.get('codegen') and not result['codegen'].get('error'):
            generated_code = result['codegen']['code']

        output_data.append({str(test_name_prefix): generated_code})
    else:
        output_data.append({str(test_name_prefix): "NO_GEN_CODE_AVAILABLE"})

output_filename = "_gen_code_for_T1_to_T10_tests.json"
with open(output_filename, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"Generated code for tests T1-T10 written to {output_filename}")


In [ ]:
# --- View generated code for a specific test ---

idx = 1
if test_results[idx].get('codegen'):
    print(test_results[idx]['codegen']['code'])


In [ ]:
# --- View execution result for a specific test ---
idx = 1
if test_results[idx].get('execution'):
    print(json.dumps(test_results[idx]['execution'].get('result', {}),
          indent=2, default=str)[:2000])
